## Test Ticker Samples

In [ ]:


%load_ext autoreload
%autoreload 1

%aimport openqdata
%aimport openqdata.biz_calendar
%aimport openqdata.arbfree.arbfree
%aimport openqdata.widb
%aimport openqdata.widb.client
%aimport openqdata.widb.day
%aimport openqdata.widb.arbfree
%aimport openqdata.widb.options_filter
%aimport openqdata.widb.timeseries

from datetime import date
from openqdata.widb.timeseries import build_time_series, TimeSeriesDefinition, ExpiryFilter, RawDayFilter, OptionsFilter, DateRange, PoolConfig, table_of_time_series, MkArbFree
from openqdata.biz_calendar import next_business_day
from cdxcore.util import fmt_date, fmt_dict
from cdxcore.verbose import Context
from IPython.display import display

def test():
    ticker = "SPX"
    time_series_def = TimeSeriesDefinition(
        raw_day_filter = RawDayFilter(),
        expiry_filter = ExpiryFilter(),
        mk_arb_free = MkArbFree(),
        options_filter = OptionsFilter()
    )
    date_range = DateRange(start=date(2015, 4, 1), end=date(2025, 7, 1))
    pool_config = PoolConfig(num_workers=4, threading=False)
    verbose = Context.all

    samples = build_time_series(ticker, date_range, include_fields=None, exclude_fields=None, time_series_def=time_series_def, pool_config=pool_config, verbose=verbose, override_cache_mode="on" )
    return samples

samples = test()
verbose = Context.all
verbose.write(str(table_of_time_series(samples)))
print("/done")

In [ ]:
import numpy as np
from cdxcore.dynaplot import figure, colors_tableau

DEFAULT_EXPIRIES = np.array([1./255., 3./255., 5./255., 10./255., 20./255., 40./255.,0.5,1.,2.], dtype=np.float64)

def z_strike(samples):

    ks  = np.zeros((len(samples['dates']), len(DEFAULT_EXPIRIES), 2), dtype=np.float64)
    nms = np.zeros((len(samples['dates']), len(DEFAULT_EXPIRIES), 2), dtype=np.float64)

    for idd, d in enumerate(samples['dates']):

        src_exp  = samples['tte_y'][idd,:]
        expiries = np.unique(src_exp)
        strikes  = samples['pure_strike'][idd,:]
        nms_atm  = samples['nms_atm'][idd,:]
        masks    = [ expiry == src_exp for expiry in expiries ]
        strikes  = [ strikes[mask] for mask in masks ]
        nms_atm  = [ nms_atm[mask] for mask in masks ]

        lo_k     = np.array([ np.min(strikes[t]) for t in range(len(expiries)) ])
        hi_k     = np.array([ np.max(strikes[t]) for t in range(len(expiries)) ])
        lo_z     = np.array([ np.min(nms_atm[t]) for t in range(len(expiries)) ])
        hi_z     = np.array([ np.max(nms_atm[t]) for t in range(len(expiries)) ])

        ks[idd,:,0]    = np.interp( DEFAULT_EXPIRIES, expiries, lo_k )
        ks[idd,:,1]    = np.interp( DEFAULT_EXPIRIES, expiries, hi_k )
        nms[idd,:,0]   = np.interp( DEFAULT_EXPIRIES, expiries, lo_z )
        nms[idd,:,1]   = np.interp( DEFAULT_EXPIRIES, expiries, hi_z )

    with figure("External strikes", row_size=6, col_size=12, columns=2) as fig:
        axK, axZ = fig.add_subplots("Strikes", "Z-strikes")
        colors   = colors_tableau()

        for j, expiry in enumerate(DEFAULT_EXPIRIES):
            expiry = expiry * 252.
            color  = next(colors)
            axK.plot( samples['dates'], ks[:,j,0], label=f"{expiry:.1f}d min", color=color )
            axK.plot( samples['dates'], ks[:,j,1], label=f"{expiry:.1f}d max", color=color )
            axZ.plot( samples['dates'], nms[:,j,0], label=f"{expiry:.1f}Y min", color=color )
            axZ.plot( samples['dates'], nms[:,j,1], label=f"{expiry:.1f}Y max", color=color )
        axK.legend()
        axZ.legend()
        axK.tick_params(axis="x", labelrotation=90)   # vertical
        axZ.tick_params(axis="x", labelrotation=90)   # vertical

z_strike(samples)
